In [1]:
import sys

import uuid
import kfp
from google.cloud import aiplatform

sys.path.append("src")

In [2]:
PIPELIE_NAME = "iris-pipeline"
PIPELINE_ROOT = "gs://worshop-vertex-pip/pipeline"
GCS_BUCKET = 'gs://worshop-vertex-pip/pipeline'

PROJECT_ID = 'capable-hash-432501-a6'
LOCATION = 'us-east4'
BQ_DATASET = 'iris_test'
BQ_TABLE = 'iris_data'

In [3]:
@kfp.dsl.pipeline(name=PIPELIE_NAME, pipeline_root=PIPELINE_ROOT)
def pipeline(project_id: str, location: str, bq_dataset: str, bq_table: str):
    from components.data import load_data
    from components.evaluation import choose_best_model
    from components.models import decision_tree, random_forest
    from components.register import upload_model

    data_op = load_data(
        project_id=project_id, bq_dataset=bq_dataset, bq_table=bq_table
    ).set_display_name("Load data from BigQuery")

    dt_op = decision_tree(
        train_dataset=data_op.outputs["train_dataset"]
    ).set_display_name("Decision Tree")

    rf_op = random_forest(
        train_dataset=data_op.outputs["train_dataset"]
    ).set_display_name("Random Forest")

    choose_model_op = choose_best_model(
        test_dataset=data_op.outputs["test_dataset"],
        decision_tree_model=dt_op.outputs["output_model"],
        random_forest_model=rf_op.outputs["output_model"],
    ).set_display_name("Select best Model")

    upload_model(
        project_id=project_id,
        location=location,
        model=choose_model_op.outputs["best_model"],
    ).set_display_name("Register Model")


In [4]:
# Inicializa Vertex AI en la región deseada
aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,  # Asegúrate de definir la región aquí
)

In [5]:
# Genera un job_id único usando uuid
job_id = f"iris-pipeline-job-{uuid.uuid4()}"  # Esto garantiza que sea único


In [6]:
# Compila el pipeline a un archivo .yaml
kfp.compiler.Compiler().compile(
    pipeline_func=pipeline, package_path="iris_pipeline.yaml"
)

In [7]:
# Ejecuta el pipeline en Vertex AI Pipelines en la región especificada
pipeline_job = aiplatform.PipelineJob(
    display_name="iris-pipeline",
    template_path="iris_pipeline.yaml",
    job_id=job_id,  # Utiliza el nuevo job_id
    parameter_values={
        "project_id": PROJECT_ID,
        "location": LOCATION,
        "bq_dataset": BQ_DATASET,
        "bq_table": BQ_TABLE,
    },
    pipeline_root=PIPELINE_ROOT,
)


In [8]:
# Inicia el pipeline
pipeline_job.run()

Creating PipelineJob
PipelineJob created. Resource name: projects/815942598901/locations/us-east4/pipelineJobs/iris-pipeline-job-5d6be3d7-0388-48bd-9d80-c39eec10c15c
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/815942598901/locations/us-east4/pipelineJobs/iris-pipeline-job-5d6be3d7-0388-48bd-9d80-c39eec10c15c')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-east4/pipelines/runs/iris-pipeline-job-5d6be3d7-0388-48bd-9d80-c39eec10c15c?project=815942598901
PipelineJob projects/815942598901/locations/us-east4/pipelineJobs/iris-pipeline-job-5d6be3d7-0388-48bd-9d80-c39eec10c15c current state:
PipelineState.PIPELINE_STATE_RUNNING
PipelineJob run completed. Resource name: projects/815942598901/locations/us-east4/pipelineJobs/iris-pipeline-job-5d6be3d7-0388-48bd-9d80-c39eec10c15c
